In [95]:
from datasets import load_dataset
from transformers import AutoModel , AutoImageProcessor , AutoTokenizer 
from huggingface_hub import from_pretrained_keras
import cv2

import os
os.environ['CURL_CA_BUNDLE'] = ''

import re

import numpy as np
import pandas as pd
import tensorflow as tf

import datetime
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
%matplotlib inline

In [20]:
# tf.config.gpu.set_per_process_memory_growth(True)
gpus = tf.config.experimental.list_physical_devices('GPU')
for gpu in gpus:
    tf.config.experimental.set_memory_growth(gpu , True)

2024-07-19 12:45:26.118805: E tensorflow/compiler/xla/stream_executor/cuda/cuda_driver.cc:266] failed call to cuInit: CUDA_ERROR_COMPAT_NOT_SUPPORTED_ON_DEVICE: forward compatibility was attempted on non supported HW
2024-07-19 12:45:26.118829: I tensorflow/compiler/xla/stream_executor/cuda/cuda_diagnostics.cc:168] retrieving CUDA diagnostic information for host: gpu-pc
2024-07-19 12:45:26.118833: I tensorflow/compiler/xla/stream_executor/cuda/cuda_diagnostics.cc:175] hostname: gpu-pc
2024-07-19 12:45:26.118882: I tensorflow/compiler/xla/stream_executor/cuda/cuda_diagnostics.cc:199] libcuda reported version is: 535.183.1
2024-07-19 12:45:26.118900: I tensorflow/compiler/xla/stream_executor/cuda/cuda_diagnostics.cc:203] kernel reported version is: 535.171.4
2024-07-19 12:45:26.118904: E tensorflow/compiler/xla/stream_executor/cuda/cuda_diagnostics.cc:312] kernel version 535.171.4 does not match DSO version 535.183.1 -- cannot find working devices in this configuration


In [21]:
from transformers import TFAutoModelForSequenceClassification , TFAutoModel , TFAutoModelForImageClassification , TFBertModel
from tensorflow.keras.optimizers import Adam

vit_processor = AutoImageProcessor.from_pretrained("google/vit-base-patch16-224")
vit_tf_model = TFAutoModel.from_pretrained("google/vit-base-patch16-224")

/home/user/miniconda3/envs/python38/lib/python3.8/site-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/home/user/miniconda3/envs/python38/lib/python3.8/site-packages/urllib3/connectionpool.py:1061: InsecureRequestWarning: Unverified HTTPS request is being made to host 'huggingface.co'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/1.26.x/advanced-usage.html#ssl-warnings
  warnings.warn(
/home/user/miniconda3/envs/python38/lib/python3.8/site-packages/urllib3/connectionpool.py:1061: InsecureRequestWarning: Unverified HTTPS request is being made to host 'huggingface.co'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/1.26.x/advanced-usage.html#ssl-warnings
  warnings.warn(
/home/user/miniconda3/en

In [93]:
def get_intersection(res , ans):
    c = -1
    total = 0
    total_ans = 0
    flag = 0 #number of intervals that are not actually action
    for i in res:
        if c < 0 or i[0] >= ans[c][1]: #means we have passed beyond 
            c += 1
        if c >= len(ans) : break
        if i[1] <= ans[c][0]:
            flag += 1
            continue
        inter = min(i[1] , ans[c][1]) - max(i[0] , ans[c][0])
        total += inter
    for i in ans:
        total_ans += i[1] - i[0]
    
    # print("Total ans time : " , total_ans) 
    # print("Total intersecting time : " , total)
    return total/total_ans , flag

def get_timestamps(vid):
    res = []
    with open(vid, 'r') as f:
        data = f.read()
    Bs_data = BeautifulSoup(data, "xml")
    b_unique = Bs_data.find_all('time')
    for i in b_unique:
        ts = tuple(map(int , re.split(":|-" , i.text)))
        res += [ts]
    return res

In [24]:
idx = 1 #video number

frames = []
video = cv2.VideoCapture(f"./inputs/full/v_ArmFlapping_{idx:02d}.avi")
xml = f"SSBD/annotations/v_ArmFlapping_{idx:02d}.xml"

ans = get_timestamps(xml)

while True:
    read, frame= video.read()
    if not read:
        break
    frames.append(frame)
frames = np.array(frames)

frames.shape , ans

((1098, 224, 224, 3), [(18, 24), (27, 32)])

In [25]:
img_feat = vit_processor(frames , return_tensors = "np")
img_feat["pixel_values"].shape

(1098, 3, 224, 224)

Creating a custom double ended queue class

In [26]:
class WINDOW:
    def __init__(self , length):
        self.length = length
        self.mean = 0
        self.arr = np.zeros(shape=(0,768))
    def __len__(self):
        return self.length
    def __str__(self):
        return str(self.arr)
    def __call__(self):
        return self.arr
    def push(self , item):
        self.arr = np.insert(self.arr , 0 , item , axis=0)
        if len(self.arr) > self.length:
            self.arr = self.arr[:-1]
    
window = WINDOW(7)
window()

array([], shape=(0, 768), dtype=float64)

Testing the new class

In [27]:
# window.push(x_vit.numpy()[:1])
# print(window().shape)
# window.push(x_vit.numpy()[:1])
# print(window().shape)
# window.push(x_vit.numpy()[:1])
# print(window().shape)
# window.push(x_vit.numpy()[:1])
# print(window().shape)
# window.push(x_vit.numpy()[:1])
# print(window().shape)
# window.push(x_vit.numpy()[:1])
# print(window().shape)
# window.push(x_vit.numpy()[:1])
# print(window().shape)
# window.push(x_vit.numpy()[:1])
# print(window().shape)
# window.push(x_vit.numpy()[:1])
# print(window().shape)
# window.push(x_vit.numpy()[:1])
# print(window().shape)

In [32]:
WINDOW_LENGTH = 7
win = WINDOW(WINDOW_LENGTH)
avg = np.zeros(shape=(768,))
final_frames = []
res = []
THRESHOLD = 100
THRESHOLD_INTERSECTION = 0.6 #should be 60% intersection or more
fps = video.get(cv2.CAP_PROP_FPS)

prev_sec = -1
prev_start = -1
threshold_sec = 0.5 #500 milliseconds
for i in range(len(frames)):
    print(f"\rProcessing {i+1}/{len(frames)}" , end="")
    x = np.array(img_feat["pixel_values"][i:i+1])
    x_vit = vit_tf_model(x).pooler_output
    
    _diff = np.abs(np.subtract(avg , x_vit))
    diff = np.sum(_diff)
    
    win.push(x_vit)
    avg = win().mean(axis=0)
    # print(diff)
    if diff > THRESHOLD:
        final_frames.append(frames[i])
        sec = (i+1) / fps
        # ti = str(datetime.timedelta(seconds=sec))
        # print("\tAdded frame at ", ti)
    
    if sec - prev_sec > threshold_sec:
        # ti = str(datetime.timedelta(seconds=prev_sec))
        # print("\t till ", ti)
        # print("-"*50)
        # ti = str(datetime.timedelta(seconds=sec))
        # print("+"*50)
        # print("\t\tAdded frame at ", ti)
        res += [(round(prev_start , 2) , round(prev_sec , 2))]
        prev_start = sec        
        
    prev_sec = sec

tp , fp = get_intersection(res , ans)
tp = 1 if tp > THRESHOLD_INTERSECTION else 0
fp = min(fp , 1)
        
vid_out = cv2.VideoWriter(f"./TEST_{idx}.avi", cv2.VideoWriter_fourcc(*'MP4V'), video.get(cv2.CAP_PROP_FPS),
                            (224,224))
for f in final_frames:
    vid_out.write(f)
vid_out.release()

print("DONE")

Processing 1/1098	 till  -1 day, 23:59:59
--------------------------------------------------
++++++++++++++++++++++++++++++++++++++++++++++++++
		Added frame at  0:00:00.041459
Processing 45/1098	 till  0:00:00.995025
--------------------------------------------------
++++++++++++++++++++++++++++++++++++++++++++++++++
		Added frame at  0:00:01.865672
Processing 81/1098	 till  0:00:02.404643
--------------------------------------------------
++++++++++++++++++++++++++++++++++++++++++++++++++
		Added frame at  0:00:03.358209
Processing 163/1098	 till  0:00:06.177446
--------------------------------------------------
++++++++++++++++++++++++++++++++++++++++++++++++++
		Added frame at  0:00:06.757877
Processing 217/1098	 till  0:00:07.628524
--------------------------------------------------
++++++++++++++++++++++++++++++++++++++++++++++++++
		Added frame at  0:00:08.996683
Processing 237/1098	 till  0:00:08.996683
--------------------------------------------------
++++++++++++++++++++++++

OpenCV: FFMPEG: tag 0x5634504d/'MP4V' is not supported with codec id 12 and format 'avi / AVI (Audio Video Interleaved)'
OpenCV: FFMPEG: fallback to use tag 0x34504d46/'FMP4'


In [36]:
get_intersection(res , ans)

(0.7754545454545455, 10)

In [8]:
from bs4 import BeautifulSoup

with open('SSBD/annotations/v_ArmFlapping_01.xml', 'r') as f:
    data = f.read()

# Passing the stored data inside
# the beautifulsoup parser, storing
# the returned object 
Bs_data = BeautifulSoup(data, "xml")

# Finding all instances of tag 
# `unique`
b_unique = Bs_data.find_all('time')
for i in b_unique:
    print(i.text)

18:24
27:32


In [7]:
import xml.etree.ElementTree as ET

# Passing the path of the
# xml document to enable the
# parsing process
tree = ET.parse('SSBD/annotations/v_ArmFlapping_01.xml')

# getting the parent tag of
# the xml document
root = tree.getroot()

# printing the root (parent) tag
# of the xml document, along with
# its memory location
print(root)

# printing the attributes of the
# first tag from the parent 
print(root.get("behaviours"))


<Element 'video' at 0x7a4e06fc8680>
None


In [14]:
# res = [(0,1) , (5,10) , (11,12) , (18,28) , (29,32)]
# ans = [(18,24) , (27,32)]

# def get_intersection(res , ans):
#     c = -1
#     total = 0
#     total_ans = 0
#     for i in res:
#         if i[1] <= ans[c][0]:
#             continue
#         if c < 0 or i[0] >= ans[c][1]: #means we have passed beyond 
#             c += 1
#             total_ans += ans[c][1] - ans[c][0]
#         inter = min(i[1] , ans[c][1]) - max(i[0] , ans[c][0])
#         total += inter
    
#     # print("Total ans time : " , total_ans) 
#     # print("Total intersecting time : " , total)
#     return total/total_ans
    
# get_intersection(res , ans)

0.8181818181818182

### Complete Code

In [94]:
WINDOW_LENGTH = 7

# idx = 1 #video number

FP = 0 #false positives
TP = 0 #true positives

DIR = "inputs/full"
vids = os.listdir(DIR)

for vid in vids:
    frames = []
    name = os.path.splitext(vid)[0]
    print("="*25 , f"{name}" , "="*25)
    video = cv2.VideoCapture(f"{DIR}/{name}.avi")
    xml = f"SSBD/annotations/{name}.xml"

    ans = get_timestamps(xml)

    while True:
        read, frame= video.read()
        if not read:
            break
        frames.append(frame)
    frames = np.array(frames)

    win = WINDOW(WINDOW_LENGTH)
    avg = np.zeros(shape=(768,))
    final_frames = []
    res = []
    THRESHOLD = 100
    THRESHOLD_INTERSECTION = 0.6 #should be 60% intersection or more
    fps = video.get(cv2.CAP_PROP_FPS)

    prev_sec = -1
    prev_start = -1
    threshold_sec = 0.5 #500 milliseconds

    img_feat = vit_processor(frames , return_tensors = "np")

    for i in range(len(frames)):
        print(f"\rProcessing {i+1}/{len(frames)}" , end="")
        x = np.array(img_feat["pixel_values"][i:i+1])
        x_vit = vit_tf_model(x).pooler_output
        
        _diff = np.abs(np.subtract(avg , x_vit))
        diff = np.sum(_diff)
        
        win.push(x_vit)
        avg = win().mean(axis=0)
        # print(diff)
        if diff > THRESHOLD:
            final_frames.append(frames[i])
            sec = (i+1) / fps
            # ti = str(datetime.timedelta(seconds=sec))
            # print("\tAdded frame at ", ti)
        
        if sec - prev_sec > threshold_sec:
            # ti = str(datetime.timedelta(seconds=prev_sec))
            # print("\t till ", ti)
            # print("-"*50)
            # ti = str(datetime.timedelta(seconds=sec))
            # print("+"*50)
            # print("\t\tAdded frame at ", ti)
            res += [(round(prev_start , 2) , round(prev_sec , 2))]
            prev_start = sec        
            
        prev_sec = sec

    print("\n", res , " | ", ans)
    tp , fp = get_intersection(res , ans)
    print(f"Interaction % : {tp:.2f}\tFalse Clips : {fp}")
    tp = 1 if tp > THRESHOLD_INTERSECTION else 0
    fp = min(fp , 1)
    print(f"TP : {tp}\tFP : {fp}")
    
    FP += fp
    TP += tp
            
    vid_out = cv2.VideoWriter(f"./TEST/{name}.mp4", cv2.VideoWriter_fourcc(*'MP4V'), video.get(cv2.CAP_PROP_FPS),
                                (224,224))
    for f in final_frames:
        vid_out.write(f)
    vid_out.release()

    print("DONE")

========================= v_Spinning_25 =========================


ValueError: invalid literal for int() with base 10: '0025:0028'

In [43]:
print("{:.2f}".format(97.2345))

97.23


In [80]:
res , ans = [(-1, -1), (0.07, 19.6), (20.33, 22.73), (23.47, 23.47), (24.13, 25.67), (29.67, 29.67), (32.93, 33.0)] , [(2, 14)]

def get_intersection(res , ans):
    c = -1
    total = 0
    total_ans = 0
    flag = 0 #number of intervals that are not actually action
    for i in res:
        if c < 0 or i[0] >= ans[c][1]: #means we have passed beyond 
            c += 1
        if c >= len(ans) : break
        if i[1] <= ans[c][0]:
            flag += 1
            continue
        inter = min(i[1] , ans[c][1]) - max(i[0] , ans[c][0])
        total += inter
    for i in ans:
        total_ans += i[1] - i[0]
    
    # print("Total ans time : " , total_ans) 
    # print("Total intersecting time : " , total)
    return total/total_ans , flag

get_intersection(res , ans)

(1.0, 1)

In [88]:
import os

os.listdir("inputs/full")

['v_Spinning_25.avi',
 'v_ArmFlapping_21.avi',
 'v_Spinning_08.avi',
 'v_Spinning_07.avi',
 'v_Spinning_12.avi',
 'v_Spinning_01.avi',
 'v_ArmFlapping_03.avi',
 'v_ArmFlapping_09.avi',
 'v_ArmFlapping_20.avi',
 'v_HeadBanging_05.avi',
 'v_Spinning_21.avi',
 'v_HeadBanging_01.avi',
 'v_HeadBanging_04.avi',
 'v_HeadBanging_02.avi',
 'v_Spinning_02.avi',
 'v_ArmFlapping_04.avi',
 'v_ArmFlapping_16.avi',
 'v_ArmFlapping_25.avi',
 'v_HeadBanging_19.avi',
 'v_ArmFlapping_05.avi',
 'v_ArmFlapping_12.avi',
 'v_HeadBanging_13.avi',
 'v_HeadBanging_15.avi',
 'v_Spinning_14.avi',
 'v_ArmFlapping_10.avi',
 'v_ArmFlapping_15.avi',
 'v_HeadBanging_10.avi',
 'v_Spinning_15.avi',
 'v_ArmFlapping_02.avi',
 'v_Spinning_19.avi',
 'v_Spinning_18.avi',
 'v_ArmFlapping_01.avi',
 'v_HeadBanging_20.avi',
 'v_Spinning_11.avi',
 'v_HeadBanging_18.avi',
 'v_HeadBanging_14.avi',
 'v_ArmFlapping_14.avi',
 'v_HeadBanging_06.avi',
 'v_ArmFlapping_18.avi',
 'v_HeadBanging_07.avi',
 'v_Spinning_04.avi',
 'v_ArmFlappin

In [89]:
os.path.splitext("Hey.there.txt")

('Hey.there', '.txt')